# AI Case Intelligence — RAG Evaluation (Google Colab)

This notebook evaluates a legal-judgment RAG pipeline using your own documents and manually verified test questions. It generates real metrics when you run it; it does not invent scores.

**Metrics:** Retrieval Hit Rate@K, Recall@K, answer correctness, faithfulness, hallucination rate, citation existence/support, abstention precision/recall/accuracy, and latency.

**Important:** Quality metrics require a test set with verified reference answers and annotated relevant passages. Human-review labels are required for defensible correctness, faithfulness, hallucination, and citation-support scores. Results apply to this notebook configuration, not automatically to your deployed Render app.

## 1. Install dependencies

In [ ]:
%pip -q install sentence-transformers groq pypdf pandas numpy

## 2. Configure Groq securely

Add a Colab Secret named `GROQ_API_KEY` using the key icon, or enter the key through the hidden prompt below. Never paste the key into a notebook cell. Set `MODEL_NAME` to the same supported Groq Qwen model your application uses.

In [ ]:
import os, getpass, time, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter GROQ_API_KEY (hidden): ")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is required.")
client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "qwen/qwen3-32b"  # Change to the supported model ID used by your app.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
print("Groq configured. API key is not displayed.")
print("LLM:", MODEL_NAME, "| Embeddings:", EMBEDDING_MODEL_NAME)

## 3. Upload text-based PDF judgments

In [ ]:
from google.colab import files
from pypdf import PdfReader
uploaded = files.upload()
documents = []
for filename, raw in uploaded.items():
    if not filename.lower().endswith(".pdf"):
        print(f"Skipping {filename}: this cell expects PDFs.")
        continue
    path = Path("/content") / filename
    path.write_bytes(raw)
    reader = PdfReader(str(path))
    doc_id = hashlib.sha256((filename + str(len(raw))).encode()).hexdigest()[:12]
    page_count = 0
    for page_no, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            documents.append({"document_id": doc_id, "document_name": filename,
                              "page": page_no, "text": text})
            page_count += 1
    print(f"{filename}: {page_count} non-empty pages; document_id={doc_id}")
if not documents:
    raise ValueError("No readable PDF text found. Scanned PDFs require OCR; OCR has not been performed here.")

## 4. Chunk documents and create embeddings

In [ ]:
def make_chunks(page_records, chunk_words=CHUNK_WORDS, overlap_words=CHUNK_OVERLAP_WORDS):
    chunks = []
    step = max(1, chunk_words - overlap_words)
    for p in page_records:
        words = p["text"].split()
        for start in range(0, len(words), step):
            piece = words[start:start + chunk_words]
            if not piece: break
            chunks.append({
                "chunk_id": f'{p["document_id"]}:p{p["page"]}:w{start}',
                "document_id": p["document_id"], "document_name": p["document_name"],
                "page": p["page"], "text": " ".join(piece)
            })
            if start + chunk_words >= len(words): break
    return chunks

chunks = make_chunks(documents)
if not chunks:
    raise ValueError("No chunks created.")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True)
print(f"{len(chunks)} chunks indexed in memory; embeddings shape={chunk_vectors.shape}")

## 5. Create the evaluation dataset

Prepare a CSV with one row per question. Required columns:
- `question`
- `document_id` (copy the ID printed above)
- `reference_answer` (manually verified from the judgment)
- `relevant_chunk_ids` (pipe-separated chunk IDs that contain the evidence)
- `answerable` (`true` or `false`)
- `expected_pages` (optional)

The next cell creates a blank template and displays chunks so you can annotate relevant chunk IDs. Do not use unverified model answers as reference answers.

In [ ]:
template_path = "/content/evaluation_questions_template.csv"
pd.DataFrame(columns=["question","document_id","reference_answer","relevant_chunk_ids","answerable","expected_pages"]).to_csv(template_path, index=False)
print("Template:", template_path)
print("Document IDs:")
display(pd.DataFrame(documents)[["document_id","document_name","page"]].drop_duplicates())
print("First 30 chunks for annotation:")
display(pd.DataFrame(chunks)[["chunk_id","document_id","page","text"]].head(30))
from google.colab import files
files.download(template_path)

## 6. Upload the completed evaluation CSV

In [ ]:
eval_upload = files.upload()
csv_names = [n for n in eval_upload if n.lower().endswith(".csv")]
if not csv_names: raise ValueError("Upload your completed evaluation CSV.")
eval_df = pd.read_csv(csv_names[0]).fillna("")
required = ["question","document_id","reference_answer","relevant_chunk_ids","answerable"]
missing = [c for c in required if c not in eval_df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
eval_df["answerable_bool"] = eval_df["answerable"].astype(str).str.lower().str.strip().isin(["true","1","yes","y"])
print("Questions:", len(eval_df))
display(eval_df.head())

## 7. Run retrieval and Groq Qwen generation

In [ ]:
def retrieve(question, document_id, top_k=TOP_K):
    candidates = [i for i, c in enumerate(chunks) if c["document_id"] == str(document_id)]
    if not candidates: return []
    qv = embedder.encode([question], normalize_embeddings=True, convert_to_numpy=True)[0]
    sims = chunk_vectors[candidates] @ qv
    order = np.argsort(sims)[::-1][:top_k]
    return [{**chunks[candidates[i]], "similarity": float(sims[i])} for i in order]

def generate_answer(question, retrieved):
    if not retrieved:
        return "Insufficient evidence in the uploaded judgment to answer this question.", "no_context"
    context = "\n\n".join(
        f'[Source {i+1} | Page {r["page"]} | Chunk {r["chunk_id"]}]\n{r["text"]}'
        for i, r in enumerate(retrieved))
    prompt = f'''Answer using ONLY the evidence below. Do not use outside knowledge or invent facts/citations.
If evidence is insufficient, say: "Insufficient evidence in the uploaded judgment to answer this question."
Cite page numbers from the supplied metadata.

EVIDENCE:
{context}

QUESTION: {question}
Give a concise answer with source page(s).'''
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role":"system","content":"You are a careful legal-document assistant. Ground factual claims in supplied evidence."},
            {"role":"user","content":prompt}
        ],
        temperature=0)
    return resp.choices[0].message.content.strip(), "generated"

rows = []
for idx, row in eval_df.iterrows():
    t0 = time.perf_counter()
    retrieved = retrieve(str(row["question"]), str(row["document_id"]), TOP_K)
    try:
        answer, status = generate_answer(str(row["question"]), retrieved)
        error = ""
    except Exception as e:
        answer, status, error = "", "generation_error", f"{type(e).__name__}: {e}"
    latency = time.perf_counter() - t0
    rel_ids = {x.strip() for x in str(row["relevant_chunk_ids"]).split("|") if x.strip()}
    got_ids = {r["chunk_id"] for r in retrieved}
    rows.append({
        "question_id": f"Q{idx+1:03d}", "question": row["question"],
        "document_id": str(row["document_id"]), "reference_answer": row["reference_answer"],
        "answerable": bool(row["answerable_bool"]), "generated_answer": answer,
        "retrieved_chunk_ids": "|".join(r["chunk_id"] for r in retrieved),
        "retrieved_pages": "|".join(str(r["page"]) for r in retrieved),
        "retrieved_passages": "\n\n---\n\n".join(r["text"] for r in retrieved),
        "relevant_chunk_ids": "|".join(sorted(rel_ids)),
        "hit_at_k": bool(rel_ids & got_ids) if rel_ids else "",
        "recall_at_k": (len(rel_ids & got_ids)/len(rel_ids)) if rel_ids else "",
        "latency_seconds": round(latency, 3), "generation_status": status, "error": error,
        "manual_correctness": "", "manual_faithful": "",
        "manual_has_unsupported_claim": "", "manual_citations_exist": "",
        "manual_citations_supported": "", "manual_abstained": ""
    })
    print(f'{idx+1}/{len(eval_df)} {rows[-1]["question_id"]}: {status}, {latency:.1f}s')
results = pd.DataFrame(rows)
display(results[["question_id","question","generated_answer","retrieved_pages","latency_seconds","error"]])

## 8. Export results for human review

Download this CSV and inspect each generated answer against its retrieved passages and the original judgment. Fill in:
- `manual_correctness`: 1 correct, 0 incorrect, 0.5 partially correct
- `manual_faithful`: 1 if factual claims are supported by retrieved context, else 0
- `manual_has_unsupported_claim`: 1 if any factual claim is unsupported, else 0
- `manual_citations_exist`: 1 if cited page/chunk exists, else 0
- `manual_citations_supported`: 1 if citations support the claims, else 0
- `manual_abstained`: 1 if answer clearly abstains, else 0

Leave unreviewed cells blank. Then upload the annotated CSV in the next cell.

In [ ]:
review_path = "/content/rag_per_question_results_for_review.csv"
results.to_csv(review_path, index=False)
from google.colab import files
files.download(review_path)

## 9. Upload reviewed results and calculate report

In [ ]:
review_upload = files.upload()
review_csvs = [n for n in review_upload if n.lower().endswith(".csv")]
if not review_csvs: raise ValueError("Upload the annotated results CSV.")
review_df = pd.read_csv(review_csvs[0]).fillna("")
display(review_df.head())

In [ ]:
def numcol(name):
    return pd.to_numeric(review_df[name].replace("", np.nan), errors="coerce") if name in review_df else pd.Series(dtype=float)
def pct_mean(series):
    s = series.dropna()
    return round(float(s.mean()*100), 2) if len(s) else None

hit = review_df.loc[review_df["hit_at_k"].astype(str).str.lower().isin(["true","false"]), "hit_at_k"].astype(str).str.lower().map({"true":1,"false":0})
recall = pd.to_numeric(review_df["recall_at_k"], errors="coerce")
correct = numcol("manual_correctness")
faithful = numcol("manual_faithful")
halluc = numcol("manual_has_unsupported_claim")
cite_exists = numcol("manual_citations_exist")
cite_support = numcol("manual_citations_supported")
abstain = numcol("manual_abstained")
answerable = review_df["answerable"].astype(str).str.lower().isin(["true","1","yes","y"])
valid_abs = abstain.notna()
pred_abs = abstain.fillna(-1).eq(1)
unanswerable = ~answerable
tp = int((pred_abs & unanswerable & valid_abs).sum())
fp = int((pred_abs & answerable & valid_abs).sum())
fn = int((~pred_abs & unanswerable & valid_abs).sum())
tn = int((~pred_abs & answerable & valid_abs).sum())
prec = tp/(tp+fp) if tp+fp else None
rec = tp/(tp+fn) if tp+fn else None
acc = (tp+tn)/(tp+tn+fp+fn) if tp+tn+fp+fn else None
lat = pd.to_numeric(review_df["latency_seconds"], errors="coerce").dropna()
report = {
 "project":"AI Case Intelligence", "llm_model":MODEL_NAME, "embedding_model":EMBEDDING_MODEL_NAME,
 "top_k":TOP_K, "total_questions":int(len(review_df)),
 "retrieval_hit_rate_at_k_percent":pct_mean(hit),
 "retrieval_hit_rate_denominator":int(len(hit)),
 "recall_at_k_percent":pct_mean(recall),
 "recall_at_k_denominator":int(recall.notna().sum()),
 "answer_correctness_percent":pct_mean(correct),
 "answer_correctness_reviewed_n":int(correct.notna().sum()),
 "faithfulness_percent":pct_mean(faithful),
 "faithfulness_reviewed_n":int(faithful.notna().sum()),
 "hallucination_rate_percent":pct_mean(halluc),
 "hallucination_reviewed_n":int(halluc.notna().sum()),
 "citation_existence_accuracy_percent":pct_mean(cite_exists),
 "citation_existence_reviewed_n":int(cite_exists.notna().sum()),
 "citation_support_accuracy_percent":pct_mean(cite_support),
 "citation_support_reviewed_n":int(cite_support.notna().sum()),
 "abstention_precision_percent":round(prec*100,2) if prec is not None else None,
 "abstention_recall_percent":round(rec*100,2) if rec is not None else None,
 "abstention_accuracy_percent":round(acc*100,2) if acc is not None else None,
 "abstention_reviewed_n":int(valid_abs.sum()),
 "mean_latency_seconds":round(float(lat.mean()),3) if len(lat) else None,
 "latency_measured_n":int(len(lat)),
 "note":"Null means not measured. Human-review metrics include only rows with valid review labels."
}
print(json.dumps(report, indent=2))
with open("/content/rag_evaluation_summary.json","w") as f: json.dump(report,f,indent=2)
review_df.to_csv("/content/rag_per_question_results.csv",index=False)
with open("/content/rag_evaluation_report.md","w") as f:
    f.write("# AI Case Intelligence — RAG Evaluation Report\n\n")
    f.write("Generated from this run and human-reviewed labels. Null metrics mean not measured.\n\n```json\n")
    f.write(json.dumps(report,indent=2))
    f.write("\n```\n")
print("Created summary JSON, per-question CSV, and Markdown report.")

In [ ]:
from google.colab import files
files.download("/content/rag_evaluation_summary.json")
files.download("/content/rag_per_question_results.csv")
files.download("/content/rag_evaluation_report.md")

## 10. What to show your evaluator

Show a question, the retrieved passages, the answer with source pages, the reviewed per-question CSV, and the aggregate report. State the number of questions tested and how reference answers/relevant passages were verified.

**Limitations:** This notebook runs its own retrieval/generation pipeline and does not by itself prove that the deployed Render app behaves identically. Scanned PDFs may require OCR. Faithfulness, unsupported claims, citation support, and answer correctness require human review for defensible results. Do not claim scores before executing the notebook and completing the annotations.